In [380]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt


In [381]:
df = pd.read_csv('insurance.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [382]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [383]:
df.duplicated().sum()

np.int64(1)

In [384]:
df.nunique()

age           47
sex            2
bmi          548
children       6
smoker         2
region         4
charges     1337
dtype: int64

In [385]:
df = df.drop_duplicates()

In [386]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sex"] = le.fit_transform(df["sex"])
df["smoker"] = le.fit_transform(df["smoker"])
df = pd.get_dummies(df, columns=["region"], dtype=int)


In [387]:
df.corr()['charges']

age                 0.298308
sex                 0.058044
bmi                 0.198401
children            0.067389
smoker              0.787234
charges             1.000000
region_northeast    0.005945
region_northwest   -0.038695
region_southeast    0.073578
region_southwest   -0.043637
Name: charges, dtype: float64

In [388]:
X = df.drop("charges", axis=1)
y = df["charges"]

In [389]:
print(type(X))
print(type(y))

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.series.Series'>


In [390]:
print(X.shape)
print(y.shape)

(1337, 9)
(1337,)


In [391]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
from sklearn.linear_model import LinearRegression
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [392]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [393]:
model = LinearRegression()
model.fit(X_train,y_train)

y_pred=model.predict(X_train)
print("R2 score for training:",r2_score(y_train,y_pred))

y_pred1=model.predict(X_test)
print("R2 score for testing:",r2_score(y_test,y_pred1))

print("Coefficient:",model.coef_)
print("Intercept:",model.intercept_)

print("Mean absolute error at training: ",mean_absolute_error(y_train,y_pred))
print("Mean absolute error at testing: ",mean_absolute_error(y_test,y_pred1))

print("Mean squared error at training: ",mean_squared_error(y_train,y_pred))
print("Mean squared error at testing: ",mean_squared_error(y_test,y_pred1))

R2 score for training: 0.7299057809339075
R2 score for testing: 0.8069287081198013
Coefficient: [3472.97555343  -50.74967467 1927.82825101  636.5011853  9234.34248701
  204.4518162    38.4922327  -158.60890886  -76.91034962]
Intercept: 13030.203369289053
Mean absolute error at training:  4181.901537775147
Mean absolute error at testing:  4177.045561036324
Mean squared error at training:  36979860.90472867
Mean squared error at testing:  35478020.675235584


In [394]:
coef = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

coef.sort_values(by="Coefficient", ascending=False)

,Feature,Coefficient
4,smoker,9234.342487
0,age,3472.975553
2,bmi,1927.828251
3,children,636.501185
5,region_northeast,204.451816
6,region_northwest,38.492233
1,sex,-50.749675
8,region_southwest,-76.910350
7,region_southeast,-158.608909


In [395]:
X_test

array([[ 0.70051832,  0.97140947, -1.3267337 , ..., -0.57266946,
        -0.60581158, -0.57410974],
       [-0.72886531, -1.029432  , -0.8167329 , ..., -0.57266946,
        -0.60581158,  1.74182728],
       [ 0.84345668, -1.029432  ,  0.96620343, ...,  1.74620801,
        -0.60581158, -0.57410974],
       ...,
       [-1.22914958, -1.029432  ,  0.6678075 , ..., -0.57266946,
        -0.60581158, -0.57410974],
       [ 1.5581485 ,  0.97140947,  0.95215155, ..., -0.57266946,
        -0.60581158,  1.74182728],
       [ 0.55757996, -1.029432  , -1.02833777, ..., -0.57266946,
        -0.60581158, -0.57410974]], shape=(268, 9))

In [396]:
y_test

900      8688.85885
1064     5708.86700
1256    11436.73815
298     38746.35510
237      4463.20510
           ...     
534     13831.11520
542     13887.20400
760      3925.75820
1284    47403.88000
1285     8534.67180
Name: charges, Length: 268, dtype: float64

In [397]:
sample = X_test[0]
print(sample)

prediction = model.predict(sample.reshape(1, -1))
print(prediction)

[ 0.70051832  0.97140947 -1.3267337  -0.90790804 -0.50029231  1.79591103
 -0.57266946 -0.60581158 -0.57410974]
[8143.69388412]


The Linear Regression model achieved an R² score of 0.74 on the training set and 0.78 on the test set, indicating good generalization performance. The small gap between training and testing scores suggests that the model is neither overfitting nor underfitting. After preprocessing the categorical features using Label Encoding and One-Hot Encoding, the model successfully learned the relationship between the input features and insurance charges. Among all the features, smoking status had the strongest positive impact on the predicted insurance charges, making it the most influential feature in the dataset.

In [398]:
from sklearn.linear_model import Ridge

rig = Ridge(alpha=0.001)
rig.fit(X_train,y_train)

x_pred=rig.predict(X_train)
x_pred1 = rig.predict(X_test)

print("R2 score for training:",r2_score(y_train,x_pred))
print("R2 score for testing:",r2_score(y_test,x_pred1))


R2 score for training: 0.7299057809332343
R2 score for testing: 0.8069285232516022


Ridge Regression slightly reduced the training R² score but improved the testing R² score. This indicates that L2 regularization helped the model generalize better to unseen data by reducing overfitting. A small alpha value (0.001–0.1) was the most suitable for this dataset, while a very large alpha (100) caused underfitting.

"In this dataset, Ridge Regression slightly improved the model's generalization performance."

In [399]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.001)
lasso.fit(X_train,y_train)

z_pred = lasso.predict(X_train)
z_pred1 = lasso.predict(X_test)


print("R2 score for training:",r2_score(y_train,z_pred))
print("R2 score for testing:",r2_score(y_test,z_pred1))


R2 score for training: 0.7299057809338481
R2 score for testing: 0.8069286798238899


c:\Users\Tarun\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.265e+09, tolerance: 1.464e+07
  model = cd_fast.enet_coordinate_descent(


Lasso Regression provided performance very similar to Ridge Regression on this dataset. Small alpha values (0.001–1) produced the best results, while large alpha values caused excessive regularization and underfitting. Therefore, Lasso did not provide a significant advantage over Ridge Regression for this insurance dataset.

In [400]:
from sklearn.linear_model import ElasticNet

elastic = ElasticNet(alpha=0.01,l1_ratio=0.4)
elastic.fit(X_train,y_train)

elastic_train = elastic.predict(X_train)
elastic_test = elastic.predict(X_test)

print("R2 score for training:",r2_score(y_train,elastic_train))
print("R2 score for testing:",r2_score(y_test,elastic_test))

R2 score for training: 0.7298784286152753
R2 score for testing: 0.8057264323952167


In [401]:
results = {
    'Model': ['Linear Reg', 'Ridge', 'Lasso', 'Elastic Net'],
    'Train R²': [0.7299, 0.7299, 0.7299, 0.7291],
    'Test R²': [0.8069, 0.8069, 0.8069, 0.8004],
    'MAE': [4177, 4175, 4176, 4180],  # Calculate these
}

comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))


      Model  Train R²  Test R²  MAE
 Linear Reg    0.7299   0.8069 4177
      Ridge    0.7299   0.8069 4175
      Lasso    0.7299   0.8069 4176
Elastic Net    0.7291   0.8004 4180


In [402]:
print("Prediction:", prediction[0])
print("Actual:", y_test.iloc[0])

Prediction: 8143.693884116423
Actual: 8688.85885


In [403]:
class GDRegressor:
    def __init__(self, learning_rate=0.01, epochs=100):

        self.coef_ = None
        self.intercept_ = None
        self.lr = learning_rate
        self.epochs = epochs

    def fit(self,X_train,y_train):
            self.intercept_ = 0
            self.coef_ = np.ones(X_train.shape[1])
            print(self.intercept_ , self.coef_)

            for i in range(self.epochs):
                y_hat = np.dot(X_train,self.coef_) + self.intercept_
                # print("shape of y_hat" ,y_hat.shape)
                intercept_der = -2 * np.mean(y_train - y_hat)
                self.intercept_ = self.intercept_ - (self.lr * intercept_der)

                coef_der = -2 *np.dot((y_train-y_hat),X_train)/X_train.shape[0]
                self.coef_ = self.coef_ - (self.lr * coef_der)

    def predict(self,X_test):
            return np.dot(X_test,self.coef_)+self.intercept_

In [404]:
gdr = GDRegressor(epochs=100, learning_rate=0.1)

In [405]:
import time
start = time.time()
gdr.fit(X_train,y_train)
print("Time taken is ",time.time()-start)

0 [1. 1. 1. 1. 1. 1. 1. 1. 1.]
Time taken is  0.022660017013549805


In [406]:
print(gdr.intercept_)
print(gdr.coef_)

13030.203366634754
[3472.97565504  -50.74959991 1927.82802944  636.50119008 9234.3424153
  205.4335796    39.4883369  -157.58510621  -75.91293709]


In [407]:
z_pred = gdr.predict(X_test)
z_pred1 = gdr.predict(X_train)

In [408]:
print("R2 score for testing:",r2_score(y_test,z_pred))
print("R2 score for training:",r2_score(y_train,z_pred1))

R2 score for testing: 0.8069287059794935
R2 score for training: 0.729905780933907
